In [1]:
# install PySpark
! pip install pyspark >& /dev/null

In [2]:
# create Spark Session and Spark Context
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
spark = SparkSession.builder.appName('Final Project').getOrCreate()

In [3]:
#Upload the first dataset
transactions = spark.read.csv('2024 Units by Product Specific Code and Order Num.csv',
                           header = True,
                           inferSchema = True)

In [4]:
#show the top of the uploaded data
transactions.show()

+-------------+------------+-----+-------------+-------------+------------+-------------+--------+----+--------+------+---+
|   Brand Name|Sold to Code|State|         Type|          SKU|Order Number|Product Specific Code|Flag BSP|Size|   Color| Sales|Qty|
+-------------+------------+-----+-------------+-------------+------------+-------------+--------+----+--------+------+---+
|      Brand 8|      155243|   CA|     Sunglass|      CT0428S|  1201527099|  30014705001|       -|  54|    GOLD|1918.0|  1|
|      Brand 8|      166778|   CA|     Sunglass|      CT0428S|  1201527101|  30014705001|       -|  54|    GOLD|1918.0|  1|
|Brand 2|      171458|   MD|Optical Frame|   SL 599 OPT|  1201549162|  30014298003|       -|  58|   BROWN| 132.0|  1|
|      Brand 8|      147108|   IL|Optical Frame|      CT0033O|  1201575257|  30002236001|     BSP|  53|   BLACK| 478.0|  1|
|         Brand 10|      155648|   IL|Optical Frame|      PU0378O|  1201575628|  30013331003|       -|  53|   GREEN| 51.92|  1|
| 

In [5]:
#create the color code columnm with the last three digits of the product specific code column
transactions = transactions.withColumn(
       "Color Code",
       f.substring("Product Specific Code", -3, 3)  # Extract last 3 characters
   )

transactions.show()

+-------------+------------+-----+-------------+-------------+------------+-------------+--------+----+--------+------+---+----------+
|   Brand Name|Sold to Code|State|         Type|          SKU|Order Number|Product Specific Code|Flag BSP|Size|   Color| Sales|Qty|Color Code|
+-------------+------------+-----+-------------+-------------+------------+-------------+--------+----+--------+------+---+----------+
|      Brand 8|      155243|   CA|     Sunglass|      CT0428S|  1201527099|  30014705001|       -|  54|    GOLD|1918.0|  1|       001|
|      Brand 8|      166778|   CA|     Sunglass|      CT0428S|  1201527101|  30014705001|       -|  54|    GOLD|1918.0|  1|       001|
|Brand 2|      171458|   MD|Optical Frame|   SL 599 OPT|  1201549162|  30014298003|       -|  58|   BROWN| 132.0|  1|       003|
|      Brand 8|      147108|   IL|Optical Frame|      CT0033O|  1201575257|  30002236001|     BSP|  53|   BLACK| 478.0|  1|       001|
|         Brand 10|      155648|   IL|Optical Frame| 

In [6]:
# Filter out rows where 'Color' is 'none'
transactions = transactions.filter(~f.col('Color').isin(['none']))

# Filter out rows where 'Size' is 0
transactions = transactions.filter(~f.col('Size').isin([0]))

# Filter out rows where 'Sales' are less than 1
transactions = transactions.filter(f.col('Sales') >= 1)


transactions.show()

+-------------+------------+-----+-------------+-------------+------------+-------------+--------+----+--------+------+---+----------+
|   Brand Name|Sold to Code|State|         Type|          SKU|Order Number|Product Specific Code|Flag BSP|Size|   Color| Sales|Qty|Color Code|
+-------------+------------+-----+-------------+-------------+------------+-------------+--------+----+--------+------+---+----------+
|      Brand 8|      155243|   CA|     Sunglass|      CT0428S|  1201527099|  30014705001|       -|  54|    GOLD|1918.0|  1|       001|
|      Brand 8|      166778|   CA|     Sunglass|      CT0428S|  1201527101|  30014705001|       -|  54|    GOLD|1918.0|  1|       001|
|Brand 2|      171458|   MD|Optical Frame|   SL 599 OPT|  1201549162|  30014298003|       -|  58|   BROWN| 132.0|  1|       003|
|      Brand 8|      147108|   IL|Optical Frame|      CT0033O|  1201575257|  30002236001|     BSP|  53|   BLACK| 478.0|  1|       001|
|         Brand 10|      155648|   IL|Optical Frame| 

In [7]:
#performs a GroupBy and Count on 'Brand Name' in descending order
transactions\
  .groupBy('Brand Name')\
  .count()\
  .orderBy(f.desc('count'))\
  .show()

+-----------------+-----+
|       Brand Name|count|
+-----------------+-----+
|            Brand 1|28736|
|    Brand 2| 9194|
|        Brand 7| 2751|
|             Brand 10| 2254|
|            Brand 3| 1827|
|          Brand 8| 1080|
|Brand 5|  950|
|       Brand 6|  834|
|   Brand 4|  674|
|          Brand 9|  196|
|              Brand 11|   10|
+-----------------+-----+



In [8]:
#remove rows for Lindberg and Brand 11

# Filter out rows where 'Brand Name' is Brand 11 or Lindberg
transactions = transactions.filter(~f.col('Brand Name').isin(['Brand 11', 'Lindberg']))

# Display the filtered DataFrame
transactions.show()

+-------------+------------+-----+-------------+-------------+------------+-------------+--------+----+--------+------+---+----------+
|   Brand Name|Sold to Code|State|         Type|          SKU|Order Number|Product Specific Code|Flag BSP|Size|   Color| Sales|Qty|Color Code|
+-------------+------------+-----+-------------+-------------+------------+-------------+--------+----+--------+------+---+----------+
|      Brand 8|      155243|   CA|     Sunglass|      CT0428S|  1201527099|  30014705001|       -|  54|    GOLD|1918.0|  1|       001|
|      Brand 8|      166778|   CA|     Sunglass|      CT0428S|  1201527101|  30014705001|       -|  54|    GOLD|1918.0|  1|       001|
|Brand 2|      171458|   MD|Optical Frame|   SL 599 OPT|  1201549162|  30014298003|       -|  58|   BROWN| 132.0|  1|       003|
|      Brand 8|      147108|   IL|Optical Frame|      CT0033O|  1201575257|  30002236001|     BSP|  53|   BLACK| 478.0|  1|       001|
|         Brand 10|      155648|   IL|Optical Frame| 

In [9]:
#re performs the Groupby and count on the 'Brand Name' in descending order

transactions\
  .groupBy('Brand Name')\
  .count()\
  .orderBy(f.desc('count'))\
  .show()

+-----------------+-----+
|       Brand Name|count|
+-----------------+-----+
|            Brand 1|28736|
|    Brand 2| 9194|
|        Brand 7| 2751|
|             Brand 10| 2254|
|            Brand 3| 1827|
|          Brand 8| 1080|
|Brand 5|  950|
|       Brand 6|  834|
|   Brand 4|  674|
|          Brand 9|  196|
+-----------------+-----+



In [10]:
#Performs a GroupBy and count on the 'Flag BSP (Best Selling Product)' column in desceding order

transactions\
  .groupBy('Flag BSP (Best Selling Product)')\
  .count()\
  .orderBy(f.desc('count'))\
  .show()

+--------+-----+
|Flag BSP|count|
+--------+-----+
|       -|38244|
|     BSP|10252|
+--------+-----+



In [11]:
#Performs a GroupBy and count on the 'State' column in descending order

transactions\
  .groupBy('State')\
  .count()\
  .orderBy(f.desc('count'))\
  .show()

+-----+-----+
|State|count|
+-----+-----+
|   CA|11730|
|   TX| 6279|
|   FL| 5247|
|   IL| 4638|
|   WA| 2123|
|   NY| 1806|
|   NC| 1420|
|   IN| 1279|
|   NJ| 1252|
|   VA|  978|
|   AZ|  946|
|   OH|  916|
|   MI|  916|
|   GA|  879|
|   PA|  724|
|   NV|  706|
|   MA|  564|
|   TN|  498|
|   MD|  497|
|   KS|  486|
+-----+-----+
only showing top 20 rows



In [12]:
#Import packages for functions and FPGrowth

from pyspark.sql import functions as f
from pyspark.ml.fpm import FPGrowth

# 1. Prepare the data for FPGRowth, Group by Order Number and collect Brand Names into a list
transactions_grouped = transactions.groupBy("Order Number").agg(f.collect_list("Product Specific Code").alias("items"))

# 2. create a DataFrame from the list of grouped items by order number
transactions_df = transactions_grouped.select("items")

# 3. Apply FP-Growth algorithm
# Adjust minSupport and minConfidence as needed
fpGrowth = FPGrowth(minSupport=0.005, minConfidence=0.4)

# 4. Fit the transactions_df to the FPGrowth algorithm
model = fpGrowth.fit(transactions_df)

# 5. Get frequent itemsets and association rules
frequent_itemsets = model.freqItemsets
association_rules = model.associationRules

# Display results
frequent_itemsets.show()
association_rules.show()

+-------------+----+
|        items|freq|
+-------------+----+
|[30015506004]|  65|
|[30014789003]|  63|
|[30014465001]|  79|
|[30013909002]|  75|
|[30014116004]|  50|
|[30014496001]|  55|
|[30014950001]|  53|
|[30008139001]|  52|
|[30014949001]|  82|
|[30014447003]|  61|
|[30014651005]|  57|
|[30008538003]|  71|
|[30014446004]|  70|
|[30014795003]|  67|
|[30014860003]|  78|
|[30013236002]| 141|
|[30013725001]|  75|
|[30011265008]|  66|
|[30007771011]|  69|
|[30014810002]|  52|
+-------------+----+
only showing top 20 rows

+--------------------+-------------+-------------------+------------------+--------------------+
|          antecedent|   consequent|         confidence|              lift|             support|
+--------------------+-------------+-------------------+------------------+--------------------+
|[30013229004, 300...|[30001750005]| 0.4585987261146497|11.217684526039715|0.007214428857715431|
|[30001750005, 300...|[30013229004]| 0.5070422535211268|10.369429692911567|0.00721

In [13]:
# Create a mapping of Product Specific Code to SKU, Color Code, Flag BSP, Color, Size
material_code_sku_mapping = transactions.select("Product Specific Code", "SKU", "Color Code","Flag BSP","Color","Size") \
                                                .rdd \
                                                .map(lambda row: (row[0], row[1] + " " + row[2] + " " + row[3] + " " + row[4] + " " + str(row[5]))) \
                                                .collectAsMap()

# Define a funtion to replace Product Specific Code with SKU
def replace_with_sku(items):
    return [material_code_sku_mapping.get(item, item) for item in items]  # Get SKU from mapping, or keep original if not found

replace_udf = f.udf(replace_with_sku, f.ArrayType(f.StringType()))

# Apply the UDF to the 'items' column of frequent_itemsets
frequent_itemsets_with_sku = frequent_itemsets.withColumn("items", replace_udf(f.col("items")))

# Apply the UDF to the 'items' column of association_Rules
association_rules_with_sku = association_rules.withColumn("antecedent", replace_udf(f.col("antecedent"))) \
                                              .withColumn("consequent", replace_udf(f.col("consequent")))

In [14]:
# Show the results of the FPGRowth

frequent_itemsets_with_sku.show()
association_rules_with_sku.show()

+--------------------+----+
|               items|freq|
+--------------------+----+
|[GG1717O 004 - HA...|  65|
|[GG1510O 003 - HA...|  63|
|[GG1465OA 001 - B...|  79|
|[GG1291O 002 - GO...|  75|
|[SL 579 004 - BRO...|  50|
|[GG1466OA 001 - B...|  55|
|[GG1506OJ 001 - B...|  53|
|[GG0613O 001 - BL...|  52|
|[GG1537OK 001 - B...|  82|
|[GG1448O 003 - SI...|  61|
|[SL 622 005 - HAV...|  57|
|[SL M63 003 BSP G...|  71|
|[GG1447O 004 - GR...|  70|
|[GG1518O 003 - GR...|  67|
|[GG1589O 003 - VI...|  78|
|[GG0034SN 002 BSP...| 141|
|[SL 554 001 BSP B...|  75|
|[SL 458 008 BSP G...|  66|
|[SL 296 011 BSP H...|  69|
|[GG1530O 002 - HA...|  52|
+--------------------+----+
only showing top 20 rows

+--------------------+--------------------+-------------------+------------------+--------------------+
|          antecedent|          consequent|         confidence|              lift|             support|
+--------------------+--------------------+-------------------+------------------+------------

In [15]:
#Export frequent item sets to CSV
frequent_itemsets_with_sku.toPandas().to_csv('frequent_itemsets.csv', index=False)

# Export association rules to CSV
association_rules_with_sku.toPandas().to_csv('association_rules.csv', index=False)

In [16]:
#filter out Brand 1 rows
filtered_transactions_noBR1 = transactions.filter(~f.col('Brand Name').isin(['Brand 1']))

# Display the filtered DataFrame
filtered_transactions_noBR1.show()

+-------------+------------+-----+-------------+-------------+------------+-------------+--------+----+--------+------+---+----------+
|   Brand Name|Sold to Code|State|         Type|          SKU|Order Number|Product Specific Code|Flag BSP|Size|   Color| Sales|Qty|Color Code|
+-------------+------------+-----+-------------+-------------+------------+-------------+--------+----+--------+------+---+----------+
|      Brand 8|      155243|   CA|     Sunglass|      CT0428S|  1201527099|  30014705001|       -|  54|    GOLD|1918.0|  1|       001|
|      Brand 8|      166778|   CA|     Sunglass|      CT0428S|  1201527101|  30014705001|       -|  54|    GOLD|1918.0|  1|       001|
|Brand 2|      171458|   MD|Optical Frame|   SL 599 OPT|  1201549162|  30014298003|       -|  58|   BROWN| 132.0|  1|       003|
|      Brand 8|      147108|   IL|Optical Frame|      CT0033O|  1201575257|  30002236001|     BSP|  53|   BLACK| 478.0|  1|       001|
|         Brand 10|      155648|   IL|Optical Frame| 

In [17]:
#Redo the algorithm without Brand 1 Rows

# 1. Prepare the data for FPGRowth, Group by Order Number and collect Brand Names into a list
transactions_grouped_noBR1 = filtered_transactions_noBR1.groupBy("Order Number").agg(f.collect_list("Product Specific Code").alias("items"))

# 2. create a DataFrame from the list of grouped items by order number
transactions_df_noBR1 = transactions_grouped_noBR1.select("items")

# 3. Apply FP-Growth algorithm
# Adjust minSupport and minConfidence as needed
fpGrowth = FPGrowth(minSupport=0.004, minConfidence=0.4)

# 4. Fit the transactions_df to the FPGrowth algorithm
model_noBR1 = fpGrowth.fit(transactions_df_noBR1)

# 5. Get frequent itemsets and association rules
frequent_itemsets_noBR1 = model_noBR1.freqItemsets
association_rules_noBR1 = model_noBR1.associationRules

# Display results
frequent_itemsets_noBR1.show()
association_rules_noBR1.show()

+-------------+----+
|        items|freq|
+-------------+----+
|[30011497001]|  26|
|[30016135001]|  25|
|[30015355002]|  20|
|[30012300004]|  19|
|[30002733003]|  32|
|[30000469007]|  30|
|[30014621001]|  23|
|[30015018003]|  21|
|[30014667002]|  22|
|[30009337015]|  21|
|[30014170008]|  22|
|[30014751001]|  18|
|[30014388001]|  32|
|[30008529001]|  25|
|[30013565003]|  24|
|[30011021003]|  18|
|[30014757002]|  29|
|[30014123002]|  28|
|[30014636003]|  27|
|[30010900004]|  19|
+-------------+----+
only showing top 20 rows

+-------------+-------------+-------------------+------------------+--------------------+
|   antecedent|   consequent|         confidence|              lift|             support|
+-------------+-------------+-------------------+------------------+--------------------+
|[30000027005]|[30002036007]|  0.425531914893617|  35.0786308973173|0.005274261603375527|
|[30014129001]|[30000839003]|0.42857142857142855|12.598006644518271|0.005537974683544304|
|[30014388001]|[3001

In [18]:
# re edit the product specific codes to the fields specified


# Apply the UDF to the 'items' column of frequent_itemsets
frequent_itemsets_with_sku_noBR1 = frequent_itemsets_noBR1.withColumn("items", replace_udf(f.col("items")))

# Apply the UDF to the 'items' column of association_Rules
association_rules_with_sku_noBR1 = association_rules_noBR1.withColumn("antecedent", replace_udf(f.col("antecedent"))) \
                                              .withColumn("consequent", replace_udf(f.col("consequent")))

In [19]:
# Show the results of the FPGRowth

frequent_itemsets_with_sku_noBR1.show()
association_rules_with_sku_noBR1.show()

+--------------------+----+
|               items|freq|
+--------------------+----+
|[AM0346O 001 BSP ...|  26|
|[SL 753 OPT 001 -...|  25|
|[SL 692 OPT 002 -...|  20|
|[PJ0061O 004 - GR...|  19|
|[SL 237/F 003 - G...|  32|
|[SL 39 007 - BROW...|  30|
|[SL M124 001 - BL...|  23|
|[MB0333O 003 - BL...|  21|
|[CH0217OA 002 - P...|  22|
|[SL 386 015 - RED...|  21|
|[MB0259OK 008 - G...|  22|
|[SL 640 001 - BLA...|  18|
|[PU0411O 001 - BL...|  32|
|[PU0280O 001 BSP ...|  25|
|[MB0252O 003 - GR...|  24|
|[BV1096O 003 BSP ...|  18|
|[SL 645/F 002 - H...|  29|
|[SL M116 002 - HA...|  28|
|[SL 627 003 - BEI...|  27|
|[AM0305O 004 BSP ...|  19|
+--------------------+----+
only showing top 20 rows

+--------------------+--------------------+-------------------+------------------+--------------------+
|          antecedent|          consequent|         confidence|              lift|             support|
+--------------------+--------------------+-------------------+------------------+------------

In [20]:
#Export frequent itemsets to CSV
frequent_itemsets_with_sku_noBR1.toPandas().to_csv('frequent_itemsets_noBR1.csv', index=False)

#Export association rules to CSV
association_rules_with_sku_noBR1.toPandas().to_csv('association_rules_noBR1.csv', index=False)

In [21]:
#filter out Brand 10

filtered_transactions_noBR1noBR10 = filtered_transactions_noBR1.filter(~f.col('Brand Name').isin(['Brand 10']))

# Display the filtered DataFrame
filtered_transactions_noBR1noBR10.show()

+-------------+------------+-----+-------------+-------------+------------+-------------+--------+----+--------+------+---+----------+
|   Brand Name|Sold to Code|State|         Type|          SKU|Order Number|Product Specific Code|Flag BSP|Size|   Color| Sales|Qty|Color Code|
+-------------+------------+-----+-------------+-------------+------------+-------------+--------+----+--------+------+---+----------+
|      Brand 8|      155243|   CA|     Sunglass|      CT0428S|  1201527099|  30014705001|       -|  54|    GOLD|1918.0|  1|       001|
|      Brand 8|      166778|   CA|     Sunglass|      CT0428S|  1201527101|  30014705001|       -|  54|    GOLD|1918.0|  1|       001|
|Brand 2|      171458|   MD|Optical Frame|   SL 599 OPT|  1201549162|  30014298003|       -|  58|   BROWN| 132.0|  1|       003|
|      Brand 8|      147108|   IL|Optical Frame|      CT0033O|  1201575257|  30002236001|     BSP|  53|   BLACK| 478.0|  1|       001|
|        Brand 3|      127721|   LA|Optical Frame|   

In [22]:
#Redo the algorithm without Brand 1 or Brand 10 Rows

# 1. Prepare the data for FPGRowth, Group by Order Number and collect Brand Names into a list
transactions_grouped_noBR1noBR10 = filtered_transactions_noBR1noBR10.groupBy("Order Number").agg(f.collect_list("Product Specific Code").alias("items"))

# 2. create a DataFrame from the list of grouped items by order number
transactions_df_noBR1noBR10 = transactions_grouped_noBR1noBR10.select("items")

# 3. Apply FP-Growth algorithm
# Adjust minSupport and minConfidence as needed
fpGrowth = FPGrowth(minSupport=0.004, minConfidence=0.4)

# 4. Fit the transactions_df to the FPGrowth algorithm
model_noBR1noBR10 = fpGrowth.fit(transactions_df_noBR1noBR10)

# 5. Get frequent itemsets and association rules
frequent_itemsets_noBR1noBR10 = model_noBR1noBR10.freqItemsets
association_rules_noBR1noBR10 = model_noBR1noBR10.associationRules

# Display results
frequent_itemsets_noBR1noBR10.show()
association_rules_noBR1noBR10.show()

+-------------+----+
|        items|freq|
+-------------+----+
|[30015341006]|  24|
|[30014579004]|  23|
|[30014646004]|  18|
|[30015317001]|  18|
|[30011265007]|  30|
|[30015340003]|  29|
|[30014651002]|  19|
|[30015018003]|  21|
|[30014757001]|  19|
|[30009337015]|  21|
|[30014592007]|  16|
|[30015160004]|  20|
|[30014125004]|  31|
|[30015247004]|  23|
|[30014170008]|  22|
|[30013796004]|  17|
|[30014751002]|  27|
|[30011497001]|  26|
|[30016135001]|  25|
|[30014108003]|  17|
+-------------+----+
only showing top 20 rows

+--------------------+-------------+-------------------+------------------+--------------------+
|          antecedent|   consequent|         confidence|              lift|             support|
+--------------------+-------------+-------------------+------------------+--------------------+
|[30007771011, 300...|[30001570004]| 0.5909090909090909|15.683308494783905|0.004014823965410748|
|       [30014757002]|[30014660003]| 0.4827586206896552|16.629493763756418|0.00432

In [23]:
# re edit the product specific codes to the fields specified

# Apply the UDF to the 'items' column of frequent_itemsets
frequent_itemsets_with_sku_noBR1noBR10 = frequent_itemsets_noBR1noBR10.withColumn("items", replace_udf(f.col("items")))

# Apply the UDF to the 'items' column of association_Rules
association_rules_with_sku_noBR1noBR10 = association_rules_noBR1noBR10.withColumn("antecedent", replace_udf(f.col("antecedent"))) \
                                              .withColumn("consequent", replace_udf(f.col("consequent")))

In [24]:
# Show the results of the FPGRowth

frequent_itemsets_with_sku_noBR1noBR10.show()
association_rules_with_sku_noBR1noBR10.show()

+--------------------+----+
|               items|freq|
+--------------------+----+
|[SL 699 006 - GOL...|  24|
|[MB0293OA 004 - G...|  23|
|[AM0437O 004 BSP ...|  18|
|[SL 672 001 - BLA...|  18|
|[SL 458 007 - GRE...|  30|
|[SL M134 003 - GO...|  29|
|[SL 622 002 - HAV...|  19|
|[MB0333O 003 - BL...|  21|
|[SL 645/F 001 - B...|  19|
|[SL 386 015 - RED...|  21|
|[MB0306O 007 - BL...|  16|
|[CH0241O 004 - BL...|  20|
|[SL M117 004 - NU...|  31|
|[SL 666 004 - SIL...|  23|
|[MB0259OK 008 - G...|  22|
|[SL M107/K 004 - ...|  17|
|[SL 640 002 - HAV...|  27|
|[AM0346O 001 BSP ...|  26|
|[SL 753 OPT 001 -...|  25|
|[SL 589 003 - BEI...|  17|
+--------------------+----+
only showing top 20 rows

+--------------------+--------------------+-------------------+------------------+--------------------+
|          antecedent|          consequent|         confidence|              lift|             support|
+--------------------+--------------------+-------------------+------------------+------------

In [25]:
#Export frequent itemsets to CSV
frequent_itemsets_with_sku_noBR1noBR10.toPandas().to_csv('frequent_itemsets_noBR1noBR10.csv', index=False)

#Export association rules to CSV
association_rules_with_sku_noBR1noBR10.toPandas().to_csv('association_rules_noBR1noBR10.csv', index=False)

In [26]:
#filter out Brand 2

filtered_transactions_noBR1noBR10noBR2 = filtered_transactions_noBR1noBR10.filter(~f.col('Brand Name').isin(['Brand 2']))

# Display the filtered DataFrame
filtered_transactions_noBR1noBR10noBR2.show()

+--------------+------------+-----+-------------+--------+------------+-------------+--------+----+--------+------+---+----------+
|    Brand Name|Sold to Code|State|         Type|     SKU|Order Number|Product Specific Code|Flag BSP|Size|   Color| Sales|Qty|Color Code|
+--------------+------------+-----+-------------+--------+------------+-------------+--------+----+--------+------+---+----------+
|       Brand 8|      155243|   CA|     Sunglass| CT0428S|  1201527099|  30014705001|       -|  54|    GOLD|1918.0|  1|       001|
|       Brand 8|      166778|   CA|     Sunglass| CT0428S|  1201527101|  30014705001|       -|  54|    GOLD|1918.0|  1|       001|
|       Brand 8|      147108|   IL|Optical Frame| CT0033O|  1201575257|  30002236001|     BSP|  53|   BLACK| 478.0|  1|       001|
|         Brand 3|      127721|   LA|Optical Frame| CH0200O|  1201582638|  30014686005|       -|  51|BURGUNDY|110.88|  1|       005|
|       Brand 8|      115474|   RI|Optical Frame| CT0453O|  1201583847|  

In [27]:
#Redo the algorithm without Brand 1, Brand 10 or Brand 2 Rows

# 1. Prepare the data for FPGRowth, Group by Order Number and collect Brand Names into a list
transactions_grouped_noBR1noBR10noBR2 = filtered_transactions_noBR1noBR10noBR2.groupBy("Order Number").agg(f.collect_list("Product Specific Code").alias("items"))

# 2. create a DataFrame from the list of grouped items by order number
transactions_df_noBR1noBR10noBR2 = transactions_grouped_noBR1noBR10noBR2.select("items")

# 3. Apply FP-Growth algorithm
# Adjust minSupport and minConfidence as needed
fpGrowth = FPGrowth(minSupport=0.007, minConfidence=0.4)

# 4. Fit the transactions_df to the FPGrowth algorithm
model_noBR1noBR10noBR2 = fpGrowth.fit(transactions_df_noBR1noBR10noBR2)

# 5. Get frequent itemsets and association rules
frequent_itemsets_noBR1noBR10noBR2 = model_noBR1noBR10noBR2.freqItemsets
association_rules_noBR1noBR10noBR2 = model_noBR1noBR10noBR2.associationRules

# Display results
frequent_itemsets_noBR1noBR10noBR2.show()
association_rules_noBR1noBR10noBR2.show()

+--------------------+----+
|               items|freq|
+--------------------+----+
|       [30006911005]|  12|
|       [30013547002]|  12|
|       [30015116003]|  15|
|       [30014669001]|  14|
|       [30014579001]|  16|
|       [30014968006]|  12|
|       [30013371003]|  14|
|       [30014669004]|  13|
|       [30015155002]|  13|
|       [30014595003]|  15|
|       [30013565003]|  24|
|       [30015790004]|  14|
|       [30011966002]|  13|
|       [30015762003]|  13|
|       [30013504003]|  16|
|[30013504003, 300...|  12|
|       [30014085008]|  15|
|       [30015162001]|  12|
|       [30006759008]|  18|
|       [30013546002]|  19|
+--------------------+----+
only showing top 20 rows

+-------------+-------------+------------------+------------------+--------------------+
|   antecedent|   consequent|        confidence|              lift|             support|
+-------------+-------------+------------------+------------------+--------------------+
|[30013504003]|[30010023007]|      

In [28]:
# re edit the product specific codes to the fields specified

# Apply the UDF to the 'items' column of frequent_itemsets
frequent_itemsets_with_sku_noBR1noBR10noBR2 = frequent_itemsets_noBR1noBR10noBR2.withColumn("items", replace_udf(f.col("items")))

# Apply the UDF to the 'items' column of association_Rules
association_rules_with_sku_noBR1noBR10noBR2 = association_rules_noBR1noBR10noBR2.withColumn("antecedent", replace_udf(f.col("antecedent"))) \
                                              .withColumn("consequent", replace_udf(f.col("consequent")))

In [29]:
# Show the results of the FPGRowth

frequent_itemsets_with_sku_noBR1noBR10noBR2.show()
association_rules_with_sku_noBR1noBR10noBR2.show()

+--------------------+----+
|               items|freq|
+--------------------+----+
|[MB0042O 005 - BL...|  12|
|[MB0233O 002 - GR...|  12|
|[MB0328O 003 - BL...|  15|
|[CH0218OA 001 - B...|  14|
|[MB0293OA 001 - B...|  16|
|[MB0315OA 006 - G...|  12|
|[CH0115O 003 - BL...|  14|
|[CH0218OA 004 - G...|  13|
|[CH0238O 002 - HA...|  13|
|[MB0309OA 003 - G...|  15|
|[MB0252O 003 - GR...|  24|
|[CH0276OA 004 - G...|  14|
|[BB0198O 002 - GO...|  13|
|[MB0364O 003 - GR...|  13|
|[BB0238O 003 BSP ...|  16|
|[BB0238O 003 BSP ...|  12|
|[CH0163O 008 - BR...|  15|
|[CH0232O 001 - GO...|  12|
|[MB0011O 008 - GR...|  18|
|[MB0232O 002 - RU...|  19|
+--------------------+----+
only showing top 20 rows

+--------------------+--------------------+------------------+------------------+--------------------+
|          antecedent|          consequent|        confidence|              lift|             support|
+--------------------+--------------------+------------------+------------------+---------------

In [30]:
#Export frequent itemsets to CSV
frequent_itemsets_with_sku_noBR1noBR10noBR2.toPandas().to_csv('frequent_itemsets__noBR1noBR10noBR2.csv', index=False)

#Export association rules to CSV
association_rules_with_sku_noBR1noBR10noBR2.toPandas().to_csv('association_rules__noBR1noBR10noBR2.csv', index=False)